# Entity-Grounded Image Generation (Kaggle runner)

**Before running (one-time):**
1. Right panel **Session options**: Accelerator = **GPU T4 x2**, Internet = **On** (your Kaggle account must be phone-verified).
2. **Add-ons > Secrets**: add `GEMINI_API_KEY` and `WIKI_CONTACT` (your email or a URL), and attach them to this notebook.
3. **Code**: on your laptop run `python scripts/make_kaggle_zip.py`, upload `dist/egig_code.zip` as a new Kaggle *Dataset*,
   then **Add Input** that dataset here. (Or put a git URL in `REPO_URL` below.)

First run downloads ~14 GB of model weights (a few minutes on Kaggle); they stay cached for the session.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 1) Get the code into /kaggle/working/egig_project
import glob, os, shutil, subprocess

REPO_URL = "https://github.com/itzPEKOS/entity-grounded-image-generation.git"  # empty = use the uploaded dataset
WORK = "/kaggle/working/egig_project"

if REPO_URL:
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:  # private repo: add a GitHub token as the Kaggle secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    except Exception:
        token = ""  # public repo: no token needed
    if token:  # token goes in an HTTP header via env vars: never in the URL, .git/config, or error output
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
    if os.path.exists(os.path.join(WORK, ".git")):
        cmd = ["git", "-C", WORK, "pull", "--ff-only"]  # get your latest push
    else:
        cmd = ["git", "clone", "--depth", "1", REPO_URL, WORK]
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if proc.returncode != 0:
        hint = ("GITHUB_TOKEN was found but GitHub refused it: check it can read this repo and has not expired."
                if token else
                "Repo is private (or the URL is wrong). Make it public, or add a GitHub token as the Kaggle "
                "secret GITHUB_TOKEN, attach it to this notebook, and run this cell again.")
        raise RuntimeError(proc.stderr.strip() + "\n\n-> " + hint)
    print(subprocess.run(["git", "-C", WORK, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
else:
    hits = glob.glob("/kaggle/input/**/egig/__init__.py", recursive=True)
    assert hits, "No code found: upload dist/egig_code.zip as a Dataset and Add Input it to this notebook."
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("code in", WORK, "->", sorted(os.listdir(WORK)))

In [ ]:
# 2) Packages. torch+CUDA is preinstalled; upgrade diffusers AND transformers together so their
#    huggingface_hub requirements stay compatible (tested with diffusers 0.43 + transformers 5.24)
!pip install -q -U "diffusers>=0.32" "transformers>=4.45" "accelerate>=0.30" "google-genai>=1.0" python-dotenv
import torch, transformers, diffusers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| diffusers", diffusers.__version__,
      "| GPUs:", torch.cuda.device_count())

In [ ]:
# 3) Secrets -> environment variables
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["GEMINI_API_KEY", "WIKI_CONTACT"]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print(name, "loaded")
    except Exception:
        print(name, "MISSING: add it under Add-ons > Secrets and attach it to this notebook")
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"

In [ ]:
# 4) Check key, models, internet, GPUs
!python -m egig doctor

Optional: `smoke` loads SDXL + IP-Adapter + ControlNet + Grounding DINO once and makes two small test images.
Run it the first time to make sure everything loads (and to fill the model cache).

In [ ]:
!python -m egig smoke

In [ ]:
# 5) Build the agents once; models stay loaded between runs
import sys
sys.path.insert(0, WORK)
from egig.config import Settings
from egig.orchestrator import build_orchestrator

settings = Settings.from_env()          # change e.g. Settings.from_env(max_attempts=2, image_size=896)
orch = build_orchestrator(settings)

In [ ]:
# 6) Run the whole multi-agent pipeline
PROMPT = "A Shiba Inu sitting in front of Wat Arun at sunset"
result = orch.run(PROMPT)

In [ ]:
# 7) Show the result and every attempt
from IPython.display import Image as IPImage, Markdown, display

display(IPImage(filename=str(result.final_image), width=640))
display(Markdown((result.run_dir / "final_report.md").read_text(encoding="utf-8")))
for rep in result.reports:
    print(f"--- attempt {rep.attempt}: {rep.verdict.explanation}")
    display(IPImage(filename=str(result.run_dir / f"attempt_{rep.attempt}" / "detections.png"), width=420))

In [ ]:
# 8) Zip this run to download it (Output tab on the right)
archive = shutil.make_archive(f"/kaggle/working/run_{result.run_dir.name}", "zip", result.run_dir)
print(archive)